# Real station data -> cleaning -> a first 24-hour temperature model
Open this file in VS Code and use **Run Cell**, or open the accompanying notebook.
Select the project's `.venv` interpreter. No Python basics are repeated here.

We use NOAA GHCNh station observations for Delhi Safdarjung, Mumbai airport,
and Bengaluru HAL airport. These represent stations, not city-wide averages.
Public source: https://www.ncei.noaa.gov/products/global-historical-climatology-network-hourly
The downloaded 2026 snapshot ends 21 October. Archive publication has a delay.
Start with Mumbai because its exact-hour temperature coverage is the best.


In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT/'src/india_weather/data.py').exists():
    ROOT = ROOT.parent
if not (ROOT/'src/india_weather/data.py').exists():
    raise RuntimeError('Open the india-temperature-forecast folder in VS Code first.')
sys.path.insert(0, str(ROOT/'src'))
from india_weather.data import STATIONS, download, read_raw, clean_reports, prepare
from india_weather.model import load_hourly, make_features, validation_folds, new_model, predict, run
from india_weather.evaluation import split_week, score

CITY = 'Mumbai'  # Change to Delhi or Bengaluru after completing this walkthrough.
YEAR = 2026
WEEK_START = '2026-10-19'  # Fixed for reproducibility, Monday in IST.


## 1. Import raw reports, preserving the original file
PSV means pipe-separated values: pandas reads it with `sep='|'`.
We initially keep every field as a string so blanks and QC code `0` survive.
Inspect the rows before interpreting any missing values or units.


In [ ]:
path = download(STATIONS[CITY], YEAR)  # Uses the saved copy unless refresh=True.
raw = read_raw(path)
print('Raw shape:', raw.shape)
raw[['DATE','temperature','temperature_Quality_Code','wind_speed',
     'wind_direction','relative_humidity','precipitation']].tail(8)


## 2. Audit before cleaning
Blank rainfall means unreported, not dry weather. QC flags depend on source.
For example source 223 uses 5=removed; some legacy sources use 5=passed.
Blank QC means no flag supplied, not that an observation is guaranteed correct.
Values are already decimal Celsius and m/s, not scaled integers.


In [ ]:
variables = ['temperature','dew_point_temperature','wind_speed','wind_direction',
             'relative_humidity','sea_level_pressure','station_level_pressure','precipitation']
raw_missing = raw[variables].eq('').mean().mul(100).round(2).sort_values(ascending=False)
print('Blank percentage in raw reports:\n', raw_missing)
print('\nTemperature QC/source counts:\n', raw.groupby(
    ['temperature_Source_Code','temperature_Quality_Code']).size())
print('\nDuplicate DATE strings:', raw.DATE.duplicated().sum())


## 3. Clean and create a regular hourly grid
`clean_reports` parses UTC timestamps, rejects flagged/invalid values and
retains exact-hour reports. It creates explicit NaN rows for absent hours.
We discard half-hour reports for this initial experiment; averaging them into
the previous hour would reveal information unavailable at that hour's origin.
Duplicates stop the pipeline so we can inspect conflicts instead of guessing.
We retain plausible extremes: high temperature alone is not corruption.


In [ ]:
hourly, audit = clean_reports(raw, STATIONS[CITY])
quality = pd.DataFrame(audit['variables'])
print('Missing hourly temperature:', audit['temperature_missing_percent'], '%')
print('Subhourly reports excluded:', audit['subhourly_reports_excluded'])
quality[['variable','raw_missing','qc_rejected_nonmissing',
         'range_rejected_nonmissing','hourly_missing_percent']]


In [ ]:
hourly.tail(8)  # Missing targets are still missing. No global fillna or dropna.


## 4. Explore the actual measurements
Look for gaps, diurnal cycles and extremes before choosing model complexity.
The annual plot covers less than a full year; it cannot establish a long-term
climate anomaly. A later experiment needs multiple years of observations.


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6))
hourly.temperature.plot(ax=axes[0], lw=.6, title=f'{CITY}: observed temperature, 2026', ylabel='°C')
quality.set_index('variable').hourly_missing_percent.plot.barh(
    ax=axes[1], title='Missing measurements on the hourly grid', xlabel='Missing (%)')
plt.tight_layout()
plt.show()


## 5. Engineer causal features
Lags describe earlier states. Rolling standard deviations describe variability;
deltas describe direction and rate of change. They do not guarantee accuracy.
A rolling window includes the current measurement and only earlier hours.
Delta endpoints must both be observed. Counts expose sparse rolling windows.
Wind is represented as east/west and north/south components. Clock and seasonal
cycles use sine/cosine. Rainfall remains available for inspection but is not a
predictor yet: these reports are too sparse to treat as a complete hourly rain series.


In [ ]:
X = make_features(hourly)
X[['now_temperature','temp_lag_24h','temp_mean_24h','temp_std_24h',
   'temp_count_24h','temp_delta_1h','temp_delta_3h','temp_delta_24h',
   'wind_u_ms','wind_v_ms','hour_sin','hour_cos']].tail(8)


## 6. Construct the label and split by time
On the regular grid `shift(-24)` supplies the unfilled temperature exactly
24 hours later. Never impute y. Fit on labels before Monday 19 October IST.
Forecast origins start Monday, so their first 24-hour targets occur Tuesday.
Features can use earlier test-week measurements; fitted weights remain frozen.


In [ ]:
y = hourly.temperature.shift(-24)
known = y.notna() & hourly.temperature.notna()
known.iloc[:168] = False  # Initial warm-up for one-week lags.
as_of = pd.Timestamp.now(tz='UTC')
train, test, start, end = split_week(hourly.index, known, as_of, WEEK_START)
target_times = hourly.index+pd.Timedelta(hours=24)
print('Training pairs:', train.sum(), '| Test pairs:', test.sum())
print('Last training target (IST):', target_times[train].max().tz_convert('Asia/Kolkata'))
print('First test origin (IST):', hourly.index[test].min().tz_convert('Asia/Kolkata'))
print('Last test target (IST):', target_times[test].max().tz_convert('Asia/Kolkata'))
assert target_times[train].max() < hourly.index[test].min()


## 7. Validate using earlier periods, then fit the first model
Three expanding-window folds end before the held-out week. Each fold only
trains on labels that occurred before its first validation origin. This is
stricter than an ordinary random split. Compare against persistence in every fold.
Our fixed small gradient-boosted tree model learns the next day's change, then
adds it to the origin temperature. Median imputation is fitted inside each fold.
Trees do not require feature scaling. Do not fit an imputer on the whole dataset.


In [ ]:
fold_results = []
for fold, fit, valid, fold_start, fold_end in validation_folds(hourly.index, known, start):
    model = new_model()
    model.fit(X.loc[fit], y.loc[fit]-hourly.temperature.loc[fit])
    forecast = predict(model, X.loc[valid], hourly.temperature.loc[valid])
    fold_results.append({'fold': fold, 'model_mae_c': score(y.loc[valid].to_numpy(), forecast)['mae_c'],
        'persistence_mae_c': score(y.loc[valid].to_numpy(), hourly.temperature.loc[valid].to_numpy())['mae_c']})
pd.DataFrame(fold_results)


In [ ]:
model = new_model()
model.fit(X.loc[train], y.loc[train]-hourly.temperature.loc[train])
forecast = predict(model, X.loc[test], hourly.temperature.loc[test])
metrics = pd.DataFrame({
    'gradient_boosting': score(y.loc[test].to_numpy(), forecast),
    'persistence': score(y.loc[test].to_numpy(), hourly.temperature.loc[test].to_numpy())})
metrics


## 8. Inspect errors, not only scores
MAE is average absolute error in Celsius; RMSE penalizes large errors more.
`within_2c` is a stated tolerance rate, not generic percentage accuracy.
On this small observed test, persistence has lower MAE in every city; ML has lower RMSE in Mumbai and Bengaluru. Keep that
baseline as our reference; do not tune repeatedly against this exposed week.
Archived QC/revisions and unknown report arrival times limit live-performance claims.


In [ ]:
errors = pd.DataFrame({'actual_c': y.loc[test], 'forecast_c': forecast,
    'persistence_c': hourly.temperature.loc[test]})
errors.index = target_times[test].tz_convert('Asia/Kolkata')
errors['absolute_error_c'] = (errors.forecast_c-errors.actual_c).abs()
print('Largest errors:\n', errors.nlargest(5, 'absolute_error_c'))
errors[['actual_c','forecast_c','persistence_c']].plot(
    figsize=(11, 4), marker='.', title=f'{CITY}: observed 24-hour test, IST', ylabel='°C')
plt.tight_layout()
plt.show()


## Your next experiment
First change CITY and compare missingness and validation behaviour. Then we
can acquire 2024-2025, inspect station changes, add archived numerical forecasts
as predictors, and test forecast correction on a fresh held-out period.
Rain/ENSO/cyclone inputs need their own timestamp and coverage audit before use.
Do not claim an improvement until it beats persistence on untouched dates.
To regenerate all saved city results from the terminal:
`python -m india_weather.data`
`python -m india_weather.model --week-start 2026-10-19`
